# Embeddings
https://platform.openai.com/docs/models#embeddings

임베딩(Embeddings)은 텍스트를 수치적으로 표현한 값으로, 두 텍스트 간의 연관성을 측정하는 데 사용된다.

임베딩은 검색, 군집화(clustering), 추천 시스템, 이상 탐지, 분류와 같은 작업에 유용하다.

**모델 및 출력 차원**

| 모델 이름                     | 설명                                                              | 출력 차원 |
|-------------------------------|-------------------------------------------------------------------|-----------|
| **text-embedding-3-large**   | 영어 및 비영어 작업 모두에서 가장 강력한 성능을 가진 모델           | 3,072     |
| **text-embedding-3-small**   | 2세대 ada 임베딩 모델보다 성능이 향상된 모델                        | 1,536     |
| **text-embedding-ada-002**   | 1세대 모델 16개를 대체하는 가장 강력한 2세대 임베딩 모델             | 1,536     |

In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)
print("임베딩 모델 : ", EMBEDDING_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini
임베딩 모델 :  text-embedding-3-small


## 임베딩 함수 생성

In [2]:
def get_embedding(text,model=EMBEDDING_MODEL):
    """텍스트 하나를 임베딩 벡터로 변환한다."""
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

sample_embedding = get_embedding("김치 볶음밥은 간단하고 든든한 한끼이다.")
print(type(sample_embedding))
print(len(sample_embedding))
print(sample_embedding[:5])

<class 'list'>
1536
[0.0086822509765625, -0.040679931640625, -0.0753173828125, -0.005863189697265625, 0.0458984375]


## 코사인 유사도

In [4]:
import numpy as np
import pandas as pd

def cosine_similarity(a,b):
    """두 벡터 a,b의 코사인 유사도를 계산"""
    a = np.array(a)
    b = np.array(b)

    # 코사인 유사도 = 두 벡터의 내적 / 두 벡터의 크기의 곱
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

sentences = [
    "김치볶음밥은 간단하고 든든한 한 끼이다.",
    "비빔밥은 여러 채소와 밥을 비벼먹는 음식이다.",
    "Python의 리스트는 여러 값을 순서대로 저장한다."
]

embeddings = [get_embedding(s) for s in sentences]

for s,e in zip(sentences,embeddings):
    print(s,round(cosine_similarity(sample_embedding,e),4))

김치볶음밥은 간단하고 든든한 한 끼이다. 0.9389
비빔밥은 여러 채소와 밥을 비벼먹는 음식이다. 0.4671
Python의 리스트는 여러 값을 순서대로 저장한다. 0.1457


## FAQ 문서 데이터 준비
작은 FAQ 문서 집합을 직접 만들고, 각 문서의 본문을 임베딩한다. 실제 RAG에서는 이 문서가 PDF, 웹페이지, 사내 문서, 고객센터 FAQ 등으로 바뀔 수 있다.

In [5]:
docs = [
    {
        "id": 1,
        "title": "배송 조회",
        "text": "주문한 상품의 배송 상태는 마이페이지의 주문 내역에서 확인할 수 있다. 운송장 번호가 등록되면 택배사 배송 추적도 가능하다.",
        "category": "배송",
    },
    {
        "id": 2,
        "title": "배송지 변경",
        "text": "상품이 아직 출고되기 전이라면 마이페이지에서 배송지를 변경할 수 있다. 이미 출고된 상품은 배송지 변경이 어렵다.",
        "category": "배송",
    },
    {
        "id": 3,
        "title": "결제 수단 변경",
        "text": "주문 완료 후에는 결제 수단을 직접 변경할 수 없다. 결제 수단을 바꾸려면 기존 주문을 취소한 뒤 다시 주문해야 한다.",
        "category": "결제",
    },
    {
        "id": 4,
        "title": "환불 처리 기간",
        "text": "주문 취소나 반품이 완료되면 환불이 진행된다. 카드 결제 환불은 카드사 사정에 따라 며칠 정도 걸릴 수 있다.",
        "category": "환불",
    },
    {
        "id": 5,
        "title": "비밀번호 재설정",
        "text": "비밀번호를 잊어버린 경우 로그인 화면의 비밀번호 찾기를 통해 이메일 인증 후 새 비밀번호를 설정할 수 있다.",
        "category": "계정",
    },
    {
        "id": 6,
        "title": "회원 탈퇴",
        "text": "회원 탈퇴는 마이페이지의 계정 관리 메뉴에서 신청할 수 있다. 탈퇴 후에는 일부 주문 내역과 쿠폰 정보가 복구되지 않는다.",
        "category": "계정",
    },
]

df = pd.DataFrame(docs)
df["embedding"] = df["text"].apply(get_embedding)
df[["id", "title", "category", "text"]]

,id,title,category,text
0,1,배송 조회,배송,주문한 상품의 배송 상태는 마이페이지의 주문 내역에서 확인할 수 있다. 운송장 번호...
1,2,배송지 변경,배송,상품이 아직 출고되기 전이라면 마이페이지에서 배송지를 변경할 수 있다. 이미 출고된...
2,3,결제 수단 변경,결제,주문 완료 후에는 결제 수단을 직접 변경할 수 없다. 결제 수단을 바꾸려면 기존 주...
3,4,환불 처리 기간,환불,주문 취소나 반품이 완료되면 환불이 진행된다. 카드 결제 환불은 카드사 사정에 따라...
4,5,비밀번호 재설정,계정,비밀번호를 잊어버린 경우 로그인 화면의 비밀번호 찾기를 통해 이메일 인증 후 새 비...
5,6,회원 탈퇴,계정,회원 탈퇴는 마이페이지의 계정 관리 메뉴에서 신청할 수 있다. 탈퇴 후에는 일부 주...


## 의미 기반 검색 함수
- 사용자 질문도 임베딩 한 뒤, 각 문서 임베딩과의 유사도를 계산한다.
- 점수가 높은 문서 순서대로 정렬한다.

In [7]:
def semantic_search(query, top_k=3):
    query_embedding = get_embedding(query)
    scores = []

    for _,row in df.iterrows():
        score = cosine_similarity(query_embedding,row["embedding"])
        scores.append(score)

    result = df.copy()
    result["score"] = scores
    return result.sort_values("score",ascending=False).head(top_k)[['title','category','text','score']]

semantic_search("주문한 상품이 지금 어디쯤 있는지 확인하고 싶어")

,title,category,text,score
0,배송 조회,배송,주문한 상품의 배송 상태는 마이페이지의 주문 내역에서 확인할 수 있다. 운송장 번호...,0.607848
1,배송지 변경,배송,상품이 아직 출고되기 전이라면 마이페이지에서 배송지를 변경할 수 있다. 이미 출고된...,0.492651
2,결제 수단 변경,결제,주문 완료 후에는 결제 수단을 직접 변경할 수 없다. 결제 수단을 바꾸려면 기존 주...,0.433438


In [ ]:
# 검색어 변경해서 테스트
query = " 배송지를 다른 주소로 변경하고 싶어"
query = "카드 결제 취소했는데 환불 언제 되는지 궁금해"
query = "회원 탈퇴 어디서 신청할 수 있어?"
query = "비밀 번호 까먹었는데 어떡하지"
semantic_search(query)

,title,category,text,score
1,배송지 변경,배송,상품이 아직 출고되기 전이라면 마이페이지에서 배송지를 변경할 수 있다. 이미 출고된...,0.424807
2,결제 수단 변경,결제,주문 완료 후에는 결제 수단을 직접 변경할 수 없다. 결제 수단을 바꾸려면 기존 주...,0.347638
0,배송 조회,배송,주문한 상품의 배송 상태는 마이페이지의 주문 내역에서 확인할 수 있다. 운송장 번호...,0.325494


## mini RAG 구현
- 검색 결과를 LLM에게 전달해 답변을 생성한다. (RAG의 가장 단순한 형태)
- 규칙
    - 검색된 문서만 근거로 사용한다.
    - 근거에 없는 내용은 추측하지 않는다.
    - 답변에는 참고한 FAQ 내용을 반영한다.

In [9]:
def mini_rag_answer(question, top_k=3):
    # 사용자 질문과 의미적으로 가까운 FAQ 문서를 검색
    retrieved = semantic_search(question, top_k=top_k)

    # 검색된 문서를 LLM에게 전달할 context 문자열로 만들기
    context = "\n\n".join(
        f"[{row.title}] {row.text}" for row in retrieved.itertuples()
    )

    # LLM에게 전달할 프롬프트 구성
    prompt = f"""
다음 FAQ 정보만 근거로 사용자의 질문에 답변하세요.
FAQ에 없는 내용은 추측하지 마세요.

[FAQ 정보]
{context}

[사용자 질문]
{question}
    """

    # LLM 모델 요청
    response = client.responses.create(
        model=DEFAULT_MODEL,
        instructions="너는 쇼핑몰 고객센터 상담 도우미이다. FAQ 문서를 근거로 간결하게 답한다.",
        input=prompt,
        temperature=0.2
    )

    # 검색 결과와 생성 갑변을 함께 반환
    return retrieved, response.output_text

In [11]:
# question = "주문한 상품이 지금 어디쯤 있는지 확인하고 싶어"
# question = " 배송지를 다른 주소로 변경하고 싶어"
# question = "카드 결제 취소했는데 환불 언제 되는지 궁금해"
# question = "회원 탈퇴 어디서 신청할 수 있어?"
question = "비밀 번호 까먹었는데 어떡하지"
retrived, answer = mini_rag_answer(question)
display(retrived)
print(answer)

,title,category,text,score
4,비밀번호 재설정,계정,비밀번호를 잊어버린 경우 로그인 화면의 비밀번호 찾기를 통해 이메일 인증 후 새 비...,0.432349
0,배송 조회,배송,주문한 상품의 배송 상태는 마이페이지의 주문 내역에서 확인할 수 있다. 운송장 번호...,0.294260
5,회원 탈퇴,계정,회원 탈퇴는 마이페이지의 계정 관리 메뉴에서 신청할 수 있다. 탈퇴 후에는 일부 주...,0.269906


비밀번호를 잊으셨다면 로그인 화면의 비밀번호 찾기를 통해 이메일 인증 후 새 비밀번호를 설정할 수 있습니다.
